# 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Data Preprocessing




In [ ]:
def clean_text(text):
    """Clean text for BERT inspired by the provided notebook."""
    text = str(text).lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
cols = ['id', 'entity', 'sentiment', 'text']
train_df = pd.read_csv('twitter_training.csv', names=cols).dropna(subset=['text'])
val_full_df = pd.read_csv('twitter_validation.csv', names=cols).dropna(subset=['text'])

train_df['text'] = train_df['text'].apply(clean_text)
val_full_df['text'] = val_full_df['text'].apply(clean_text)

label_map = {val: i for i, val in enumerate(train_df['sentiment'].unique())}
train_df['label'] = train_df['sentiment'].map(label_map)
val_full_df['label'] = val_full_df['sentiment'].map(label_map)

val_df, test_df = train_test_split(val_full_df, test_size=0.5, random_state=42, stratify=val_full_df['label'])

print(f"Data Loaded. Classes: {label_map}")

# 3. Tokenization and Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

class TwitterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        encoding = tokenizer(
            str(self.texts[item]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# 4. Training Functions

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler):

    model.train()
    total_loss = 0

    for batch in dataloader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# 5. Evaluation Function

In [ ]:
def evaluate(model, dataloader):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            outputs = model(input_ids, attention_mask=mask)

            preds = torch.argmax(outputs.logits, dim=1)

            y_pred.extend(preds.cpu().numpy())
            y_true.extend(batch["labels"].numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    print("\nAccuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_map.keys(),
        yticklabels=label_map.keys()
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


# 6. EXPERIMENT FUNCTION

In [ ]:
def run_experiment(model_name, mode="full_finetune"):

    print("\n==============================")
    print("MODEL:", model_name)
    print("MODE :", mode)
    print("==============================")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label_map)
    )

    # FREEZING STRATEGIES
    if mode == "freeze_all":

        for param in model.base_model.parameters():
            param.requires_grad = False

    elif mode == "fine_tune_last_2":

        for param in model.base_model.parameters():
            param.requires_grad = False

        if hasattr(model.base_model, "encoder"):
            for layer in model.base_model.encoder.layer[-2:]:
                for param in layer.parameters():
                    param.requires_grad = True

    model.to(device)

    # DATALOADERS
    train_loader = DataLoader(
        TwitterDataset(train_df["text"], train_df["label"], tokenizer),
        batch_size=16,
        shuffle=True
    )

    val_loader = DataLoader(
        TwitterDataset(val_df["text"], val_df["label"], tokenizer),
        batch_size=16
    )

    test_loader = DataLoader(
        TwitterDataset(test_df["text"], test_df["label"], tokenizer),
        batch_size=16
    )

    optimizer = AdamW(model.parameters(), lr=2e-5)

    total_steps = len(train_loader) * 2

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    # TRAINING
    for epoch in range(2):

        model.train()
        total_loss = 0

        for batch in train_loader:

            optimizer.zero_grad()

            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                ids,
                attention_mask=mask,
                labels=labels
            )

            loss = outputs.loss

            loss.backward()

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss:", total_loss/len(train_loader))

        print("Validation Performance:")
        evaluate(model, val_loader)

    print("\nFinal Test Performance:")
    evaluate(model, test_loader)

In [ ]:
run_experiment("bert-base-uncased", "freeze_all")

run_experiment("bert-base-uncased", "fine_tune_last_2")

run_experiment("distilbert-base-uncased", "full_finetune")